**🎯  Objective **

You are building a Machine Learning system that predicts insurance risk/cost and makes it usable via an API.

👉 In simple terms:

“Given a person’s details, predict their insurance outcome (risk or charges).”

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [2]:
df = pd.read_csv('/content/insurance.csv')

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
71,38,54.1,1.81,20.25,False,Chandigarh,unemployed,Low
72,70,116.3,1.81,3.08,False,Indore,retired,High
56,24,101.9,1.55,2.86,True,Kolkata,student,Medium
62,34,72.8,1.83,35.67,False,Chennai,business_owner,Low
49,23,106.6,1.58,2.29,False,Kota,student,Medium


In [4]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [5]:
df_feat = df.copy()

In [10]:
# Feature 1: BMI
df_feat["bmi"] = df_feat["weight"] / (df_feat["height"] ** 2)

In [6]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [11]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [12]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [13]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [14]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [15]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [16]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [17]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
23,23.71,unemployed,22.187855,adult,low,2,Medium
77,0.61,retired,37.818734,senior,high,1,High
0,2.92,retired,49.227482,senior,medium,2,High
53,30.00,government_job,29.598247,adult,medium,1,Medium
29,50.00,business_owner,42.749310,senior,high,2,High


In [18]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [ ]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,medium,2,2.92000,retired
1,30.189017,adult,medium,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,young,high,1,3.34000,student
4,24.296875,senior,medium,2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,medium,1,34.01000,private_job
97,18.765432,middle_aged,low,1,44.86000,freelancer
98,30.521676,adult,medium,1,28.30000,business_owner


In [19]:
y

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High
...,...
95,Low
96,Low
97,Low
98,Low


In [20]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

("cat", OneHotEncoder(), categorical_features)

Meaning:
- "cat" → just a name (label)
- OneHotEncoder() → converts text into numbers
- categorical_features → list of categorical columns

In [21]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

## 🔍 Step-by-step explanation

### 1️⃣ `Pipeline(...)`
A **Pipeline** is a sequence of steps executed one after another.

Think of it like:

**Step 1 → Step 2 → Final Output**

---

### 2️⃣ `steps=[ ... ]`
This is a list of steps.

Each step has the form:

`(name, operation)`

---

### 3️⃣ First step: Preprocessing
```python
("preprocessor", preprocessor)

In [33]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
))
])

In [34]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
model = pipeline.fit(X_train, y_train)


In [35]:
# Predict and evaluate
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.85

In [25]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
78,27.932798,middle_aged,medium,2,14.74,freelancer
84,28.801497,senior,medium,2,0.62,retired
92,18.319942,adult,medium,2,30.00,government_job
81,31.866055,adult,high,2,22.19,freelancer
80,34.350461,middle_aged,medium,2,50.00,unemployed


In [26]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)


In [36]:
import gradio as gr
import pandas as pd
import pickle

"""
with open("model.pkl", "rb") as f:
    model = pickle.load(f)
    """

tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

all_cities = sorted(tier_1_cities + tier_2_cities)

def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

def lifestyle_risk(smoker, bmi):
    smoker_flag = str(smoker).lower() == "yes"
    if smoker_flag and bmi > 30:
        return "high"
    elif smoker_flag or bmi > 27:
        return "medium"
    else:
        return "low"

def predict_insurance(age, bmi, smoker, city, income_lpa, occupation):
    input_df = pd.DataFrame([{
        "bmi": bmi,
        "age_group": age_group(age),
        "lifestyle_risk": lifestyle_risk(smoker, bmi),
        "city_tier": city_tier(city),
        "income_lpa": income_lpa,
        "occupation": occupation
    }])

    pred = model.predict(input_df)[0]
    return f"Predicted premium category: {pred}"

demo = gr.Interface(
    fn=predict_insurance,
    inputs=[
        gr.Number(label="Age"),
        gr.Number(label="BMI"),
        gr.Dropdown(["yes", "no"], label="Smoker"),
        gr.Dropdown(all_cities, label="City"),
        gr.Number(label="Income (LPA)"),
        gr.Dropdown(
            ["retired", "freelancer", "student", "government_job", "business_owner", "unemployed", "private_job"],
            label="Occupation"
        )
    ],
    outputs="text",
    title="Insurance Prediction App"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43cb18c0babd29e874.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
